# Lecture: Finding the Evidence Before Answering

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Recite the RAG pipeline in order: chunk → embed → index → retrieve → rerank → prompt → read (§8 map).
- Explain chunking (§4) and TF-IDF/BM25/cosine (§5): why rare words discriminate and why splitting entity from fact collapses recall (0.667 → 1.000).
- Read retrieval metrics vs generation metrics (§7): recall@k/MRR blame the finder, EM blames the reader, and EM|miss ≈ 0 caps everything.


## The big idea

A language model answering from memory alone guesses — the lab's closed-book score is 0.000. RAG (retrieval-augmented generation) looks the answer up first. Split documents into short passages (**chunks**), convert each chunk and the question into number-vectors (**embeddings** such as **TF-IDF** or **BM25** weights), pull back the chunks whose vectors point the same way as the question (**cosine retrieval**, keep the **top-k**), optionally **rerank** them, paste them into the prompt, and let the reader quote rather than invent. Two failure halves stay separate: **recall@k/MRR** grade the finder, **EM (exact match)** grades the reader — and the split **EM|hit vs EM|miss** shows generation is capped by retrieval: no evidence, no answer.

Real-world analogy: open-book vs closed-book exam. Chunking tears the textbook into slips; embeddings are the index tabs; top-k pulls the 3 most promising slips; the reader copies the number off the slip. If the entity name lands on one slip and the number on another (small chunks split them), no clever reading recovers the answer.


In [ ]:
import numpy as np  # bring in numpy for the pipeline sizing demo
import matplotlib  # bring in matplotlib core to pin the headless backend
matplotlib.use('Agg')  # force headless PNG backend so any machine can render
import matplotlib.pyplot as plt  # bring in pyplot for the pipeline diagram
import matplotlib.patches as patches  # bring in patches for rounded stage boxes
np.random.seed(20)  # fix the seed so the diagram is identical every run
stages = ['docs\n48', 'chunks\n120-word', 'index\nTF-IDF', 'retrieve\ntop-3', 'rerank\nbigram', 'prompt+read\nEM']  # six pipeline stages with sizes
print('pipeline: 48 docs -> about 48 chunks (120-word) -> top-3 -> reader')  # print the main-config shape
print('small chunks (30-word) split entity from fact: recall@3 0.667 vs 1.000')  # print the chunking lesson up front
fig, ax = plt.subplots(figsize=(11, 3.0))  # create a wide canvas for six stages
ax.set_xlim(0, 12)  # fix horizontal room for boxes plus arrows
ax.set_ylim(0, 2.0)  # fix vertical room for boxes
ax.axis('off')  # hide axes because this is a diagram
ax.set_title('RAG wiring: chunk -> embed -> index -> retrieve -> rerank -> read', fontsize=11)  # title the sketch
for i in range(6):  # walk the six stages left to right
    x = 0.25 + i * 1.95  # compute the left edge of box i
    box = patches.FancyBboxPatch((x, 0.55), 1.65, 0.9, boxstyle='round,pad=0.02', facecolor='#e0e7ff', edgecolor='#4338ca')  # rounded box for this stage
    ax.add_patch(box)  # draw the box on the canvas
    ax.text(x + 0.825, 1.0, stages[i], ha='center', va='center', fontsize=8, weight='bold')  # bold stage name inside the box
    _ = i  # keep the comment habit inside the loop
for i in range(5):  # draw one arrow between each pair of neighbours
    x = 0.25 + i * 1.95  # recompute the left edge of box i
    ax.annotate('', xy=(x + 1.95, 1.0), xytext=(x + 1.7, 1.0), arrowprops=dict(arrowstyle='->', color='black', lw=1.5))  # arrow to the next box
plt.tight_layout()  # snug the layout
plt.show()  # render the pipeline diagram


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 cuts the paper, §5 writes the index tabs, §6 pulls the slips, §7 grades both halves.

| Term | One-liner | Section |
|---|---|---|
| Chunk | Short passage cut from a doc; the unit stored and returned | §4 |
| Overlap | Words shared between neighbour chunks so boundary facts survive | §4 |
| TF-IDF | frequent-here × rare-elsewhere: rare words discriminate | §5 |
| BM25 | TF-IDF-style scorer with saturated counts + length norm | §5 |
| Cosine similarity | Angle score question-to-chunk; 1.0 = same direction | §5 |
| Top-k | Keep only the k best chunks (lab uses 3) | §6 |
| Rerank | Re-sort top candidates with a 2nd rule (bigram overlap) | §6 |
| Recall@k | Fraction of questions with gold chunk in top-k | §7 |
| MRR | Mean 1/rank of the gold chunk; rewards ranking it first | §7 |
| EM | Fraction of answers with the gold number in the prediction | §7 |


### §4 Chunks and overlap — how the paper gets torn

**In one sentence: a chunker slides a word window (size 30/60/120, overlap 0/10) over each doc, and small windows split the entity sentence from the fact sentence so the question matches the wrong slip.**

The lab's trap is planted: the entity lives in sentence 1, the number in sentence 6, with filler between — a 30-word window tears them apart while a 120-word window keeps them together. Overlap stitches boundaries by sharing words between neighbours. Watch it: the tear diagram plus the recall-collapse bars (chunk-30 recall@3 = 0.667 vs chunk-120 = 1.000).


In [ ]:
import numpy as np  # reuse numpy for the recall-collapse bars
import matplotlib.pyplot as plt  # reuse pyplot for tear diagram + bars
np.random.seed(20)  # fix the seed for reproducibility
r30, r120 = 0.667, 1.000  # lab recall@3: small chunks vs whole-doc chunks
print('recall@3: chunk-30 =', r30, 'vs chunk-120 =', r120)  # print the chunking lesson
print('mechanism: 30-word windows put entity and fact on different slips')  # print why small windows fail
print('overlap 10 stitches boundaries by sharing 10 words between neighbours')  # print the stitching fix
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: tear sketch + recall bars
ax[0].axis('off')  # left panel is a diagram
ax[0].set_xlim(0, 10)  # fix horizontal room
ax[0].set_ylim(0, 6)  # fix vertical room
ax[0].set_title('Torn apart: entity | fact on separate slips')  # title the failure mode
ax[0].add_patch(plt.Rectangle((1, 3.2), 8, 1.4, facecolor='#fecaca', edgecolor='black'))  # big chunk keeps both together
ax[0].text(5, 3.9, 'chunk-120: [entity ... fact] TOGETHER', ha='center', fontsize=8)  # label the good window
ax[0].add_patch(plt.Rectangle((1, 1.6), 3.8, 1.0, facecolor='#fde68a', edgecolor='black'))  # small window holds entity only
ax[0].add_patch(plt.Rectangle((5.2, 1.6), 3.8, 1.0, facecolor='#bfdbfe', edgecolor='black'))  # small window holds fact only
ax[0].text(2.9, 2.1, 'entity slip', ha='center', fontsize=8)  # label the entity-only slip
ax[0].text(7.1, 2.1, 'fact slip', ha='center', fontsize=8)  # label the fact-only slip
ax[1].bar(['chunk-30', 'chunk-120'], [r30, r120], color=['#ef4444', '#16a34a'])  # draw the recall-collapse bars
ax[1].set_ylim(0, 1.1)  # fix the recall range with headroom
ax[1].set_ylabel('recall@3')  # label the y-axis
ax[1].set_title('Lab: big chunks reunite entity + fact')  # title the mechanism
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


### §5 TF-IDF, BM25, and cosine — why rare words win

**In one sentence: TF-IDF weights a term by log-frequency here × inverse-document-frequency elsewhere (rare = discriminative), BM25 refines it with saturated counts and length normalization, and cosine scores the angle between the L2-normalized question and chunk vectors.**

Unique entity tokens appear in exactly one chunk, so their IDF is maximal — that is why whole-doc retrieval is near-perfect here. Watch it: IDF bars (rare tower over common) plus a worked cosine where the whole chunk beats the split entity-only chunk and the unrelated chunk trails.


In [ ]:
import numpy as np  # reuse numpy for TF-IDF arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for idf + cosine bars
import math  # bring in math for the natural log in idf
np.random.seed(20)  # fix the seed for reproducibility
terms = ['vorbren0', 'arena', '34000', 'record']  # pick four probe terms from rare to common
df = np.array([1.0, 2.0, 1.0, 3.0])  # set document frequency per term over N=3 toy chunks
n = 3  # record the toy chunk count
idf = np.log((n + 1) / (df + 1)) + 1.0  # apply smoothed idf: rare terms score high
print('idf:', dict(zip(terms, np.round(idf, 3))))  # print rare terms winning
whole, split, unrel = 0.786, 0.512, 0.031  # set worked cosines: whole beats split beats unrelated
print('cosine with question: whole =', whole, '| split =', split, '| unrelated =', unrel)  # print together beats split
print('lab: unique entities carry max idf, so whole-doc recall@3 = 1.0')  # print the lab consequence
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: idf bars + cosine bars
ax[0].bar(terms, idf, color='#6366f1')  # draw idf per term
ax[0].set_ylabel('idf weight')  # label the y-axis
ax[0].set_title('Rare terms weigh more (idf)')  # title the lesson
ax[0].tick_params(axis='x', rotation=15)  # tilt the term labels
ax[0].grid(True, alpha=0.3, axis='y')  # add a light grid
ax[1].bar(['whole', 'split', 'unrelated'], [whole, split, unrel], color=['#16a34a', '#f59e0b', '#9ca3af'])  # draw cosine per chunk
ax[1].set_ylabel('cosine with question')  # label the y-axis
ax[1].set_title('Together beats split (retrieval)')  # title the mechanism
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


### §6 Top-k and rerank — keep three, re-sort with bigrams

**In one sentence: top-k keeps only the 3 highest-scoring chunks for the prompt, and the bigram reranker re-sorts those candidates by word-pair overlap with the question.**

Rerank is a second opinion, not a rescue: on well-chunked docs it barely moves (ties), but on split evidence bigram overlap promotes the entity slip (shares question words) and can bury the entity-less fact slip — the lab's chunk-30 rerank pathology drops recall@3 from 0.667 to 0.000 (measured: `retrieval.csv` chunk-30 rows, rerank 0→1; MRR 0.358→0.138). Fix the chunking, not the scorer. Watch it: a worked 5-candidate list, its top-3 cut, and the reranked order.


In [ ]:
import numpy as np  # reuse numpy for the ranking demo
import matplotlib.pyplot as plt  # reuse pyplot for the ranked-list bars
np.random.seed(20)  # fix the seed for reproducibility
cands = ['c0 gold', 'c1 entity-only', 'c2 keywords', 'c3 filler', 'c4 other-doc']  # name five toy candidates
base = np.array([0.81, 0.74, 0.52, 0.20, 0.10])  # set TF-IDF cosine per candidate with gold c0 on top
print('base ranking:', dict(zip(cands, np.round(base, 2))))  # print the retriever order
print('top-3 keeps: c0 gold, c1 entity-only, c2 keywords (gold inside -> hit)')  # print the top-k cut
print('reranked top-3: c1 entity-only, c0 gold, c2 keywords (bigrams love shared words)')  # print the second-opinion order
print('lab warning: chunk-30 + rerank drops recall@3 0.667 -> 0.000 exactly (chunk-30 rows of retrieval.csv; MRR 0.358 -> 0.138)')  # print the pathology to memorize
fig, ax = plt.subplots(figsize=(8, 4))  # make a figure for the ranked list
ax.barh(cands[::-1], base[::-1], color='#3b82f6')  # draw horizontal bars with the best on top
ax.set_xlabel('cosine score')  # label the x-axis
ax.set_title('Ranked list: top-k cuts at 3 (gold c0 must survive)')  # title the cut rule
ax.grid(True, alpha=0.3, axis='x')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the list


### §7 Recall, MRR, and EM — grade the finder and the reader separately

**In one sentence: recall@k asks 'is the gold chunk in the top-k?', MRR asks 'how high up (mean 1/rank)?', and EM asks 'does the predicted sentence contain the gold number?' — with EM|hit vs EM|miss splitting reader skill from finder luck.**

Lab-scale numbers: TF-IDF recall@3 = 1.000 with MRR ≈ 0.979 (evidence found and first), reader EM = 0.958 on whole docs vs 0.667 on chunk-30 vs 0.000 closed-book, and EM|hit ≈ 0.958 while EM|miss ≈ 0.000 — the reader cannot save a miss. Watch it: metric bars plus the hit-vs-miss split that proves retrieval caps generation.


In [ ]:
import numpy as np  # reuse numpy for the metric bars
import matplotlib.pyplot as plt  # reuse pyplot for metrics + split bars
np.random.seed(20)  # fix the seed for reproducibility
r1, r3, mrr = 0.917, 1.000, 0.979  # set representative lab retrieval: gold found and ranked first
print('retrieval: recall@1 =', r1, '| recall@3 =', r3, '| MRR =', mrr)  # print the finder grades
em_main, em_c30, em_closed = 0.958, 0.667, 0.000  # set reader EM: whole docs vs split vs no context
print('generation EM: main =', em_main, '| chunk-30 =', em_c30, '| closed-book =', em_closed)  # print the reader grades
em_hit, em_miss = 0.958, 0.000  # set the split: evidence present vs absent
print('split: EM|hit =', em_hit, '| EM|miss =', em_miss, '(no evidence, no answer)')  # print the capping proof
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: metrics + split
ax[0].bar(['recall@1', 'recall@3', 'MRR', 'EM main'], [r1, r3, mrr, em_main], color=['#3b82f6', '#16a34a', '#8b5cf6', '#f59e0b'])  # draw the four grades
ax[0].set_ylim(0, 1.1)  # fix the range with headroom
ax[0].set_ylabel('score')  # label the y-axis
ax[0].set_title('Finder (recall/MRR) vs reader (EM)')  # title the separation
ax[0].grid(True, alpha=0.3, axis='y')  # add a light grid
ax[1].bar(['EM|hit', 'EM|miss'], [em_hit, em_miss], color=['#16a34a', '#ef4444'])  # draw the split bars
ax[1].set_ylim(0, 1.1)  # fix the range with headroom
ax[1].set_ylabel('exact match')  # label the y-axis
ax[1].set_title('Retrieval caps generation (miss about 0)')  # title the takeaway
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


## How it all fits together — the system map

One RAG answer walks every § station in order. **Cut** (§4): 48 docs → 120-word chunks (30-word splits the evidence). **Index** (§5): TF-IDF/BM25 vectors, rare entities weighted most. **Retrieve + rerank** (§6): cosine top-3, bigram second opinion. **Prompt + read** (§7): paste 3 chunks, extract the number; grade finder (recall/MRR) apart from reader (EM). The map pins each term; the numbers trace one arena question through scored chunks.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.5))  # create a wide figure for the pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix horizontal placement range
ax.set_ylim(0, 4)  # fix vertical placement range
stages = [('docs\n§4', 1), ('chunks\n§4', 3.4), ('index\n§5', 5.8), ('retrieve\n§6', 8.2), ('rerank\n§6', 10.4), ('prompt->read\n§7', 12.6)]  # pipeline stations with section tags
ax.text(7, 3.6, 'RAG: find the evidence (§4-§6), then quote it (§7)', ha='center', fontsize=9, fontweight='bold')  # phase banner line
centers = [xc for _, xc in stages]  # collect the station centers for reference
for name, xc in stages:  # draw each station
    ax.text(xc, 2.4, name, ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = pipeline step
    _ = centers  # keep the comment habit inside the loop
ax.text(7, 1.2, 'GRADE finder with recall/MRR, reader with EM | EM|miss about 0 caps all', ha='center', fontsize=9, fontweight='bold', color='darkred')  # red = grading rule banner
ax.text(7, 0.5, 'closed-book EM = 0.0: memory alone guesses', ha='center', fontsize=9, style='italic')  # baseline reminder line
ax.set_title('System map: docs->chunks->index->retrieve->rerank->prompt->read')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the one-question trace
np.random.seed(20)  # fix the seed for reproducibility
chunks = ['c0 gold (entity+fact)', 'c1 entity-only', 'c2 keywords-only', 'c3 filler']  # name four toy chunks for one arena question
scores = np.array([0.81, 0.74, 0.52, 0.20])  # set cosine of the question vs each chunk
order = np.argsort(-scores)  # rank best-first
print('question: How many supporters does the arena seat? (gold number sits in c0)')  # print the traced question
print('scores:', dict(zip(chunks, np.round(scores, 2))))  # print each chunk score
print('ranked top-3 keeps gold: rank 1 -> recall@3 hit, MRR 1.0 on this question')  # print the retrieval outcome
print('reader: best sentence holds 34000 -> EM 1 | closed-book EM 0.0')  # print the generation outcome
print('lab echo: recall@3 1.0, MRR 0.979, EM main 0.958, EM|miss 0.0')  # print the lab-scale twins


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above (§8):

1. **Goal markdown** — recite chunk → embed → index → retrieve → prompt → read, and why EM|miss ≈ 0 (§7).
2. **Setup** — numpy-only seeded corpus, zero downloads; same toy IDF habit as this lecture.
3. **Tokenize + corpus + chunker** — fact sentences carry the number but NOT the entity; chunk-30 splits them, chunk-120 keeps them (§4).
4. **TF-IDF + BM25 + reader** — find `build_vocab`, `tfidf_index`, `bm25_scores`, `retrieve`, `rerank_bigram`, `read_answer` (§5–§6).
5. **Metrics + evaluate** — gold = the chunk holding the answer span; TF-IDF vs BM25 vs rerank at the main config (recall@3 = 1.000) (§7).
6. **Ablations + generation + figure** — chunk/overlap sweep (0.667 → 1.000), rerank pathology, EM main/chunk-30/closed-book (0.958/0.667/0.000); outputs land in `retrieval.csv`, `eval.csv`, `prompt_example.txt`, `rag.png`.
7. **Cleanup/next** — retrieval becomes one tool among many in P21.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Why do unique entity names get the highest IDF, and why does that matter (§5)?**

*Answer: IDF rewards terms in few chunks; a unique entity sits in exactly one, so it dominates the cosine. Whole-doc retrieval hits recall@3 = 1.000 — and splitting entity from fact (chunk-30 = 0.667) breaks exactly this link.*

2. **Rerank drops chunk-30 recall@3 from 0.667 to 0.000 (measured, not a sketch — read the two chunk-30 rows of `retrieval.csv`) — should you rerank harder (§6)?**

*Answer: no. Bigram overlap promotes chunks sharing question words (the entity slip) and buries the entity-less fact slip. Fix the chunking (bigger windows or stitching overlap), not the scorer.*

3. **EM|hit ≈ 0.958 but EM|miss ≈ 0.000 — which half failed on a miss (§7)?**

*Answer: the finder. The reader extracts well when evidence is present (0.958) and scores about 0 without it, so generation quality is capped by retrieval quality — the §8 trace shows the reader never even sees the number on a miss.*
